In [2]:
import os
import sqlite3
from logging import exception

from groq import Groq
import requests
from numpy.ma.core import empty
from simpleeval import simple_eval
from ddgs import DDGS


In [3]:
client = Groq(api_key=os.environ['llm_key'])

In [4]:
chat_completion = client.chat.completions.create(messages=[
    {
        "role": "user",
        "content": "Today we are creating an agent with ReACT.",
    }
],
    model = "llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)


ReACT (Reinforcement Learning with Abstract Contextual Temporal) is a framework for building reinforcement learning agents. To create an agent with ReACT, we'll need to break down the process into several steps:

1. **Define the environment**: Identify the problem you want the agent to solve and define the environment in which it will operate. This can include specifying the state and action spaces, rewards, and any constraints.
2. **Choose a ReACT architecture**: ReACT provides several architectures to choose from, such as ReACT-MLP (Multi-Layer Perceptron) or ReACT-LSTM (Long Short-Term Memory). Select the one that best fits your problem.
3. **Design the agent's components**: The agent will consist of several components, including:
	* **State encoder**: Maps the environment state to a compact representation.
	* **Action decoder**: Maps the agent's internal state to a specific action.
	* **Critic**: Estimates the value function, which predicts the expected return for a given state.
	*

In [5]:
def get_weather(location: str) -> str:
    weather_data = {
        "Antalya": "Sunny, 28°C",
        "Karlsruhe": "Rainy, 17°C",
        "Helsinki": "Foggy, 6°C"
    }
    return weather_data.get(location, f"No information found for the city of {location}.")


In [6]:
def get_weather_mock(location: str) -> str:
    weather_data = {"Antalya": "Sunny, 28°C", "Karlsruhe": "Rainy, 17°C"}
    return weather_data.get(location, f"No mock data for {location}.")



In [8]:
def get_weather_withAPI(location: str) -> str:
    try:
        #1.step Trying to get the geocoding
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=en&format=json"
        geo_response = requests.get(geo_url).json()

        if "results"  not in geo_response:
            return "no information found for the city of {location}."

        lat = geo_response["results"][0]["latitude"]
        lon = geo_response["results"][0]["longitude"]

        #2.step Wİth coordinates find the weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_response = requests.get(weather_url).json()

        current = weather_response["current_weather"]
        temperature = current["temperature"]
        wind_speed = current["windspeed"]


        return f"Current temperature in {location} is {temperature}°C. Wind speed is {wind_speed} km/h."

    except Exception as e:
        # just in case connections is established
        return f"API Error: {str(e)}"









In [22]:
def calculate(answer: str)-> str:
    try:
        answer = answer.strip()
        result = simple_eval(answer)
        return f"Calculated answer: {result}"

    except Exception as e:
        return f"Calculation Error: {str(e)}"



In [21]:
def search_web(query: str)-> str:
    try:
        results =  DDGS().text(query,max_results= 3 )
        if len(results)==0:
             return f"Search engine error: No results found for '{query}'"


        text = ""
        for result in results:
            title = result["title"]
            body = result["body"]
            text  += f"Title: {title}\nbody: {body}\n\n---\n\n"
        return text



    except Exception as e:
        return f"Seach engine error : {str(e)}"




In [20]:
connection = sqlite3.connect("memory.db",check_same_thread=False)
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS profile (
 key TEXT PRIMARY KEY,
 value TEXT)
  """)
connection.commit()

In [32]:
def save_memory(action_input:str) ->str:
     try:
         key,value = action_input.split(":",1)
         value = value.strip()
         key = key.strip()


         cursor.execute("INSERT OR REPLACE INTO profile(key,value) VALUES (?,?)",(key,value))
         connection.commit()
         return f"Successfully saved {key} and {value} in memory."
     except ValueError as e:
         return "Memory Error: Action Input MUST be formatted exactly as 'Key: Value'."
     except Exception as e:
         f"A problem occured while saving{str(e)}"




In [18]:
def get_user_profile(key:str)->str:
    try:
        cursor.execute("SELECT value FROM profile WHERE key = ?",(key,))
        #only show the first value
        value = cursor.fetchone()
        if value:
            return f"the value has been found from the memory: {key}= {value[0]}"
        else:
            return f"The value has been found from the memory: {key}"

    except Exception as e:
        return "An error occured while reading the memory: {str(e)}"




In [40]:
system_prompt = """
You are an AI assistant that thinks logically step-by-step. If you are asked about the weather, use the following tool:

- get_weather: Used to find out the weather of a city. It takes only the city name (e.g., "Istanbul") as a parameter.
-calculate:Used to find out the math problem that has been given such as  "25-20 " as a parameter.
-search_web:Use this function to access and search the internet for the given query such as "Which city is Turkey's Capital" as a parameter.
-save_memory: Use this to save important user details permanently. Action Input MUST be exactly in "Key: Value" format (e.g., "name: Ali Ustam" or "hobby: GTA 6"). Save only ONE key-value pair per action.
-get_user_memory: Use this method when you need to get information about the user so you can remember important details about the user such as (name)

To answer the question asked, strictly use the following format:

Thought: I am thinking step-by-step about what I need to do to answer the question and I will only use I will not do 2 action at a time .
Action: [The name of the tool to use. Only use get_weather,search_web,calculate]
Action Input: [The parameter to send to the tool]

If you used an Action, STOP here. I will provide you with the tool's result as "Observation: ...".

CRITICAL RULE: You must use ONLY ONE tool per step. Never output multiple Actions at the same time. Always wait for the [SİSTEM] Observation before taking your next action.

If you know the answer yourself or have reached the answer after an Observation, use the following format:

Thought: I know the answer.
Final Answer: [The final answer to provide to the user]
"""

In [11]:
def parse_llm_output(output_text:str):
    action = None
    action_input = None

    lines = output_text.split("\n")
    for line in lines:
        if line.startswith("Action:"):
            action = line.split("Action:")[1].strip()
        elif line.startswith("Action Input:"):
            action_input = line.split("Action Input:")[1].strip()

    return action,action_input

In [34]:
def run_reAct_agent(user_query:str,tools:dict,max_steps:int = 5 ):
    messages = [


            {"role": "system","content":system_prompt},
            {"role": "user","content":user_query}

    ]

    for step in range(max_steps):
        print(f"\n--- Adım {step + 1} ---")

        response = client.chat.completions.create(
            messages=messages,
            model="llama-3.3-70b-versatile",
            temperature=0.0
        )
        llm_output = response.choices[0].message.content
        print(llm_output)

        # 1 modelin ne düşündüğünü eski mesajlara ekle
        messages.append({"role": "assistant","content":llm_output})

        # 2 final cevaba ulaşma kontrolü
        if "Final Answer:" in llm_output:
            final_answer = llm_output.split("Final Answer:")[1].strip()
            print(final_answer)
            return final_answer


        # 3 Aracı kullnaması lazımsa veri temizleme
        action,action_input = parse_llm_output(llm_output)

        if action:
            if action in tools:
                Observation = tools[action](action_input)
                print(f"\n[SİSTEM] Observation: {Observation}")
                messages.append({"role": "user","content": f"observation: {Observation}"})
            else:
                error_msg = f"There is no tool called{action} ."
                print(f"\n[SİSTEM] Observation: {error_msg}")
                messages.append({"role": "user", "content": f"Observation: {error_msg}"})

    return "Sorry you have reached the limit."







In [35]:

my_tools_real = {"get_weather":get_weather_withAPI,"calculate":calculate,"search_web":search_web,"save_memory":save_memory,"get_user_memory":get_user_profile}
print("\nANSWER:",run_reAct_agent(" what is the weather for Karlsruhe",tools=my_tools_real))


--- Adım 1 ---
Thought: I need to find out the current weather for Karlsruhe, so I will use the get_weather tool to get the latest weather information.
Action: get_weather
Action Input: Karlsruhe

[SİSTEM] Observation: Current temperature in Karlsruhe is 27.2°C. Wind speed is 12.3 km/h.

--- Adım 2 ---
Thought: I have received the current weather information for Karlsruhe, which includes the temperature and wind speed. I can now provide this information as the answer.
Final Answer: The current temperature in Karlsruhe is 27.2°C and the wind speed is 12.3 km/h.
The current temperature in Karlsruhe is 27.2°C and the wind speed is 12.3 km/h.

ANSWER: The current temperature in Karlsruhe is 27.2°C and the wind speed is 12.3 km/h.


In [19]:
print("\nANSWER:",run_reAct_agent(" what is the sqaure root of 16",tools=my_tools_real))


--- Adım 1 ---
Thought: To find the square root of 16, I need to calculate it.
Action: calculate
Action Input: sqrt(16)

[SİSTEM] Observation: Calculation Error: Function 'sqrt' not defined, for expression 'sqrt(16)'.

--- Adım 2 ---
Thought: The calculate function does not support the sqrt function, so I need to find the square root manually or use a different approach. The square root of 16 is a number that, when multiplied by itself, gives 16. I know that 4 * 4 = 16.
Final Answer: 4
4

SONUÇ: 4


In [41]:
print("\nANSWER:",run_reAct_agent(" what is the retail price of Gta 6 for Playtation 5",tools=my_tools_real))


--- Adım 1 ---
Thought: To find the retail price of GTA 6 for PlayStation 5, I need to search the web for the latest information on the game's pricing.
Action: search_web
Action Input: GTA 6 retail price PlayStation 5

[SİSTEM] Observation: Title: GTA 6 Pre-Orders and Price Details Leak as PS Store Drops Clues
body: Reports claim new title IDs related to GTA 6 appeared in the PlayStation ... GTA 6 is officially confirmed for PlayStation 5 and Xbox Series X/S.

---

Title: Where to find a PS5 under $400 before GTA 6 launches
body: Sony's PlayStation 5 has seen two price hikes since launch: the first in August 2025 and the second back in April .

---

Title: GTA 6 PS4 Price: What You Need to Know
body: Fans are excited for GTA 6, with many wondering about the gta 6 ps4 price and gta 6 price . ... You can also buy GTA 6 online through the PlayStation ...

---



--- Adım 2 ---
Thought: The observation results did not provide a clear answer to the retail price of GTA 6 for PlayStation 5. 

In [42]:
print("\nANSWER:",run_reAct_agent(" my name is Ali  and my favourite game is GTA 6",tools=my_tools_real))



--- Adım 1 ---
Thought: I need to save the user's name in memory so I can recall it later.
Action: save_memory
Action Input: name: Ali

[SİSTEM] Observation: Successfully saved name and Ali in memory.

--- Adım 2 ---
Thought: Now that I have saved the user's name, I also need to save their favourite game in memory so I can recall it later.
Action: save_memory
Action Input: hobby: GTA 6

[SİSTEM] Observation: Successfully saved hobby and GTA 6 in memory.

--- Adım 3 ---
Thought: I know the answer, I have saved the user's details in memory.
Final Answer: Your details have been saved, your name is Ali and your hobby is GTA 6.
Your details have been saved, your name is Ali and your hobby is GTA 6.

ANSWER: Your details have been saved, your name is Ali and your hobby is GTA 6.


In [43]:
print("\nANSWER:",run_reAct_agent(" what is my name and my fav game ",tools=my_tools_real))


--- Adım 1 ---
Thought: To find out the user's name and favorite game, I need to retrieve the saved user details. I will start by trying to get the user's name.
Action: get_user_memory
Action Input: name

[SİSTEM] Observation: the value has been found from the memory: name= Ali

--- Adım 2 ---
Thought: I have found the user's name, which is Ali. Now, I need to find out the user's favorite game. I will try to retrieve the saved user details for the favorite game.
Action: get_user_memory
Action Input: hobby

[SİSTEM] Observation: the value has been found from the memory: hobby= GTA 6

--- Adım 3 ---
Thought: I know the answer.
Final Answer: Your name is Ali and your favorite game is GTA 6.
Your name is Ali and your favorite game is GTA 6.

ANSWER: Your name is Ali and your favorite game is GTA 6.


In [67]:
from ddgs import DDGS
results = DDGS().text("GTA 6 PS5 price", max_results=3)
print(results)

[{'title': 'GTA 6: Release Date, Price, Platforms, and Everything ...', 'href': 'https://bitsfrombytes.com/gta-6-release-date-price-platforms/', 'body': 'Mar 23, 2026 · Grand Theft Auto VI (GTA 6) is confirmed to release on November 19, 2026, for PlayStation 5 and Xbox Series X|S, at an expected price of $70–$80.'}, {'title': 'GTA 6 Launch And Rising Console Prices: 2026 Cost Guide', 'href': 'https://www.gamermarkt.com/blog/gta-6-launch-driving-console-prices-record-highs-2026/', 'body': 'May 8, 2026 · With GTA 6 set for November 19, 2026, Sony and Microsoft have pushed console prices to unprecedented levels. Playing GTA 6 on a new PS5 now costs more than double the GTA V launch setup, and the PS5 Pro pushes total cost near $1,000.'}, {'title': 'GTA 6 Pre-Order: Opens June 25, 2026 — Prices & Editions', 'href': 'https://leonidaverse.com/en/gta-6-preorder', 'body': 'Jun 25, 2026 · GTA 6 pre-orders open June 25, 2026. Price comparator: Standard, Special, Ultimate, Collector editions and 